In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

# Specify the path to your CSV file in Google Drive
# Example: '/content/drive/MyDrive/my_data.csv'
file_path = '/content/drive/MyDrive/QS_Master_2017_2027 (1).csv'

try:
    df = pd.read_csv(file_path)
    print(f"Data loaded successfully from {file_path}:")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please check the path and try again.")
except Exception as e:
    print(f"An error occurred while loading the file: {e}")

Data loaded successfully from /content/drive/MyDrive/QS_Master_2017_2027 (1).csv:


,university,year,country,Rank,NumericRank,overall_score,academic_reputation,employer_reputation,faculty_student,citations,sustainability
0,Massachusetts Institute of Technology (MIT),2017,USA,1,1.0,100,NaN,NaN,4.0,NaN,NaN
1,Stanford University,2017,USA,2,2.0,98.7,NaN,NaN,3.0,NaN,NaN
2,Harvard University,2017,USA,3,3.0,98.3,NaN,NaN,5.0,NaN,NaN
3,University of Cambridge,2017,UK,4,4.0,97.2,NaN,NaN,4.0,NaN,NaN
4,California Institute of Technology (Caltech),2017,USA,5,5.0,96.9,NaN,NaN,2.0,NaN,NaN


### Check for inconsistencies in 'country' names

First, let's look at the unique values and their frequencies in the `country` column to identify any potential inconsistencies.

In [ ]:
# Display unique countries and their counts
country_counts = df['country'].value_counts()
print("Unique countries and their counts:")
print(country_counts)

# Display just the unique country names
unique_countries = df['country'].unique()
print("\nList of unique country names:")
print(sorted(unique_countries))

Unique countries and their counts:
country
USA                  1941
UK                    950
China                 633
Germany               529
Japan                 499
                     ... 
Ethiopia                4
Iceland                 4
Brunei Darussalam       4
Libya                   1
Tanzania                1
Name: count, Length: 119, dtype: int64

List of unique country names:
['Argentina', 'Armenia', 'Australia', 'Austria', 'Azerbaijan', 'Bahrain', 'Bangladesh', 'Belarus', 'Belgium', 'Bolivia', 'Bosnia and Herzegovina', 'Brazil', 'Brunei', 'Brunei Darussalam', 'Bulgaria', 'Canada', 'Chile', 'China', 'Colombia', 'Costa Rica', 'Croatia', 'Cuba', 'Cyprus', 'Czech Republic', 'Czechia', 'Denmark', 'Dominican Republic', 'Ecuador', 'Egypt', 'Estonia', 'Ethiopia', 'Finland', 'France', 'Georgia', 'Germany', 'Ghana', 'Greece', 'Guatemala', 'Honduras', 'Hong Kong SAR', 'Hong Kong SAR, China', 'Hungary', 'Iceland', 'India', 'Indonesia', 'Iran (Islamic Republic of)', 'Iran, Isla

Based on the output, we can identify inconsistencies and then apply a standardization strategy. For instance, we might need to map different spellings or historical names to a common, modern country name. I will wait for the output to see if there are any specific inconsistencies to address.

### Standardizing 'country' names

Based on the observed unique values, I will create a mapping to standardize common variations and official names to more conventional or singular forms.

In [ ]:
country_mapping = {
    'Brunei': 'Brunei Darussalam',
    'Czech Republic': 'Czechia',
    'Hong Kong SAR, China': 'Hong Kong SAR',
    'Iran (Islamic Republic of)': 'Iran',
    'Iran, Islamic Republic of': 'Iran',
    'Macau SAR': 'Macao SAR, China',
    'Palestinian Territory, Occupied': 'Palestine',
    'Republic of Korea': 'South Korea',
    'Russian Federation': 'Russia',
    'Syrian Arab Republic': 'Syria',
    'Turkey': 'Türkiye',
    'Venezuela (Bolivarian Republic of)': 'Venezuela',
    'Viet Nam': 'Vietnam'
}

df['country'] = df['country'].replace(country_mapping)

print("Country names standardized. Displaying new unique countries and their counts:")
country_counts_standardized = df['country'].value_counts()
print(country_counts_standardized)

print("\nList of unique country names after standardization:")
print(sorted(df['country'].unique()))

Country names standardized. Displaying new unique countries and their counts:
country
USA           1941
UK             950
China          633
Germany        529
Japan          499
              ... 
Luxembourg       4
Iceland          4
Ethiopia         4
Libya            1
Tanzania         1
Name: count, Length: 108, dtype: int64

List of unique country names after standardization:
['Argentina', 'Armenia', 'Australia', 'Austria', 'Azerbaijan', 'Bahrain', 'Bangladesh', 'Belarus', 'Belgium', 'Bolivia', 'Bosnia and Herzegovina', 'Brazil', 'Brunei Darussalam', 'Bulgaria', 'Canada', 'Chile', 'China', 'Colombia', 'Costa Rica', 'Croatia', 'Cuba', 'Cyprus', 'Czechia', 'Denmark', 'Dominican Republic', 'Ecuador', 'Egypt', 'Estonia', 'Ethiopia', 'Finland', 'France', 'Georgia', 'Germany', 'Ghana', 'Greece', 'Guatemala', 'Honduras', 'Hong Kong SAR', 'Hungary', 'Iceland', 'India', 'Indonesia', 'Iran', 'Iraq', 'Ireland', 'Israel', 'Italy', 'Japan', 'Jordan', 'Kazakhstan', 'Kenya', 'Kuwait', 'Kyrgyz

### Checking for NULL values (missing data)

Now, let's inspect the DataFrame for any missing values across all columns. This will help us identify other areas where data cleaning might be necessary.

In [ ]:
# Check for missing values in each column
missing_values = df.isnull().sum()

# Filter to show only columns with missing values
missing_values = missing_values[missing_values > 0]

if not missing_values.empty:
    print("Columns with missing values and their counts:")
    print(missing_values)
else:
    print("No missing values found in the DataFrame.")

# Display the percentage of missing values for better understanding
print("\nPercentage of missing values per column:")
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_percentage = missing_percentage[missing_percentage > 0]
if not missing_percentage.empty:
    print(missing_percentage.round(2).astype(str) + '%')
else:
    print("No missing values found in the DataFrame.")

Columns with missing values and their counts:
Rank                     68
NumericRank              68
overall_score          3662
academic_reputation    6482
employer_reputation    6484
faculty_student         102
citations              6511
sustainability         9551
dtype: int64

Percentage of missing values per column:
Rank                    0.49%
NumericRank             0.49%
overall_score          26.33%
academic_reputation     46.6%
employer_reputation    46.62%
faculty_student         0.73%
citations              46.81%
sustainability         68.67%
dtype: object


### Adding 'Region' and 'Continent' columns

To categorize universities geographically, I will add two new columns: 'Continent' and 'Region'. This will be done by creating a mapping dictionary that assigns a continent and a more specific region to each country in the dataset.

In [ ]:
country_to_geo = {
    # North America
    'USA': {'continent': 'North America', 'region': 'Northern America'},
    'Canada': {'continent': 'North America', 'region': 'Northern America'},
    'Mexico': {'continent': 'North America', 'region': 'Central America'},
    'Cuba': {'continent': 'North America', 'region': 'Caribbean'},
    'Puerto Rico': {'continent': 'North America', 'region': 'Caribbean'},
    'Dominican Republic': {'continent': 'North America', 'region': 'Caribbean'},
    'Costa Rica': {'continent': 'North America', 'region': 'Central America'},
    'Guatemala': {'continent': 'North America', 'region': 'Central America'},
    'Honduras': {'continent': 'North America', 'region': 'Central America'},
    'Panama': {'continent': 'North America', 'region': 'Central America'},

    # South America
    'Argentina': {'continent': 'South America', 'region': 'South America'},
    'Brazil': {'continent': 'South America', 'region': 'South America'},
    'Chile': {'continent': 'South America', 'region': 'South America'},
    'Colombia': {'continent': 'South America', 'region': 'South America'},
    'Ecuador': {'continent': 'South America', 'region': 'South America'},
    'Peru': {'continent': 'South America', 'region': 'South America'},
    'Uruguay': {'continent': 'South America', 'region': 'South America'},
    'Venezuela': {'continent': 'South America', 'region': 'South America'},
    'Bolivia': {'continent': 'South America', 'region': 'South America'},
    'Paraguay': {'continent': 'South America', 'region': 'South America'},

    # Europe
    'UK': {'continent': 'Europe', 'region': 'Northern Europe'},
    'Germany': {'continent': 'Europe', 'region': 'Western Europe'},
    'France': {'continent': 'Europe', 'region': 'Western Europe'},
    'Italy': {'continent': 'Europe', 'region': 'Southern Europe'},
    'Spain': {'continent': 'Europe', 'region': 'Southern Europe'},
    'Netherlands': {'continent': 'Europe', 'region': 'Western Europe'},
    'Sweden': {'continent': 'Europe', 'region': 'Northern Europe'},
    'Switzerland': {'continent': 'Europe', 'region': 'Western Europe'},
    'Belgium': {'continent': 'Europe', 'region': 'Western Europe'},
    'Austria': {'continent': 'Europe', 'region': 'Western Europe'},
    'Denmark': {'continent': 'Europe', 'region': 'Northern Europe'},
    'Finland': {'continent': 'Europe', 'region': 'Northern Europe'},
    'Norway': {'continent': 'Europe', 'region': 'Northern Europe'},
    'Ireland': {'continent': 'Europe', 'region': 'Northern Europe'},
    'Portugal': {'continent': 'Europe', 'region': 'Southern Europe'},
    'Greece': {'continent': 'Europe', 'region': 'Southern Europe'},
    'Poland': {'continent': 'Europe', 'region': 'Eastern Europe'},
    'Czechia': {'continent': 'Europe', 'region': 'Eastern Europe'},
    'Hungary': {'continent': 'Europe', 'region': 'Eastern Europe'},
    'Romania': {'continent': 'Europe', 'region': 'Eastern Europe'},
    'Bulgaria': {'continent': 'Europe', 'region': 'Eastern Europe'},
    'Slovakia': {'continent': 'Europe', 'region': 'Eastern Europe'},
    'Croatia': {'continent': 'Europe', 'region': 'Southern Europe'},
    'Serbia': {'continent': 'Europe', 'region': 'Southern Europe'},
    'Lithuania': {'continent': 'Europe', 'region': 'Northern Europe'},
    'Latvia': {'continent': 'Europe', 'region': 'Northern Europe'},
    'Estonia': {'continent': 'Europe', 'region': 'Northern Europe'},
    'Luxembourg': {'continent': 'Europe', 'region': 'Western Europe'},
    'Malta': {'continent': 'Europe', 'region': 'Southern Europe'},
    'Cyprus': {'continent': 'Europe', 'region': 'Southern Europe'},
    'Bosnia and Herzegovina': {'continent': 'Europe', 'region': 'Southern Europe'},
    'Ukraine': {'continent': 'Europe', 'region': 'Eastern Europe'},
    'Belarus': {'continent': 'Europe', 'region': 'Eastern Europe'},
    'Iceland': {'continent': 'Europe', 'region': 'Northern Europe'},
    'Northern Cyprus': {'continent': 'Europe', 'region': 'Southern Europe'},
    'Slovenia': {'continent': 'Europe', 'region': 'Southern Europe'},

    # Asia
    'China': {'continent': 'Asia', 'region': 'Eastern Asia'},
    'Japan': {'continent': 'Asia', 'region': 'Eastern Asia'},
    'South Korea': {'continent': 'Asia', 'region': 'Eastern Asia'},
    'Taiwan': {'continent': 'Asia', 'region': 'Eastern Asia'},
    'India': {'continent': 'Asia', 'region': 'Southern Asia'},
    'Indonesia': {'continent': 'Asia', 'region': 'Southeast Asia'},
    'Malaysia': {'continent': 'Asia', 'region': 'Southeast Asia'},
    'Singapore': {'continent': 'Asia', 'region': 'Southeast Asia'},
    'Thailand': {'continent': 'Asia', 'region': 'Southeast Asia'},
    'Vietnam': {'continent': 'Asia', 'region': 'Southeast Asia'},
    'Philippines': {'continent': 'Asia', 'region': 'Southeast Asia'},
    'Brunei Darussalam': {'continent': 'Asia', 'region': 'Southeast Asia'},
    'United Arab Emirates': {'continent': 'Asia', 'region': 'Western Asia'},
    'Saudi Arabia': {'continent': 'Asia', 'region': 'Western Asia'},
    'Qatar': {'continent': 'Asia', 'region': 'Western Asia'},
    'Kuwait': {'continent': 'Asia', 'region': 'Western Asia'},
    'Oman': {'continent': 'Asia', 'region': 'Western Asia'},
    'Bahrain': {'continent': 'Asia', 'region': 'Western Asia'},
    'Israel': {'continent': 'Asia', 'region': 'Western Asia'},
    'Jordan': {'continent': 'Asia', 'region': 'Western Asia'},
    'Lebanon': {'continent': 'Asia', 'region': 'Western Asia'},
    'Iran': {'continent': 'Asia', 'region': 'Western Asia'},
    'Iraq': {'continent': 'Asia', 'region': 'Western Asia'},
    'Syria': {'continent': 'Asia', 'region': 'Western Asia'},
    'Palestine': {'continent': 'Asia', 'region': 'Western Asia'},
    'Pakistan': {'continent': 'Asia', 'region': 'Southern Asia'},
    'Bangladesh': {'continent': 'Asia', 'region': 'Southern Asia'},
    'Sri Lanka': {'continent': 'Asia', 'region': 'Southern Asia'},
    'Kazakhstan': {'continent': 'Asia', 'region': 'Central Asia'},
    'Uzbekistan': {'continent': 'Asia', 'region': 'Central Asia'},
    'Kyrgyzstan': {'continent': 'Asia', 'region': 'Central Asia'},
    'Azerbaijan': {'continent': 'Asia', 'region': 'Western Asia'},
    'Türkiye': {'continent': 'Asia', 'region': 'Western Asia'},
    'Macao SAR, China': {'continent': 'Asia', 'region': 'Eastern Asia'},
    'Hong Kong SAR': {'continent': 'Asia', 'region': 'Eastern Asia'},
    'Russia': {'continent': 'Europe', 'region': 'Eastern Europe'}, # A large portion of Russia is in Europe
    'Georgia': {'continent': 'Asia', 'region': 'Western Asia'}, # Transcontinental, but often grouped with Western Asia
    'Armenia': {'continent': 'Asia', 'region': 'Western Asia'},

    # Africa
    'Egypt': {'continent': 'Africa', 'region': 'Northern Africa'},
    'Morocco': {'continent': 'Africa', 'region': 'Northern Africa'},
    'Tunisia': {'continent': 'Africa', 'region': 'Northern Africa'},
    'South Africa': {'continent': 'Africa', 'region': 'Southern Africa'},
    'Nigeria': {'continent': 'Africa', 'region': 'Western Africa'},
    'Ghana': {'continent': 'Africa', 'region': 'Western Africa'},
    'Kenya': {'continent': 'Africa', 'region': 'Eastern Africa'},
    'Uganda': {'continent': 'Africa', 'region': 'Eastern Africa'},
    'Ethiopia': {'continent': 'Africa', 'region': 'Eastern Africa'},
    'Sudan': {'continent': 'Africa', 'region': 'Northern Africa'},
    'Libya': {'continent': 'Africa', 'region': 'Northern Africa'},
    'Tanzania': {'continent': 'Africa', 'region': 'Eastern Africa'},

    # Oceania
    'Australia': {'continent': 'Oceania', 'region': 'Australia and New Zealand'},
    'New Zealand': {'continent': 'Oceania', 'region': 'Australia and New Zealand'}
}

# Function to get continent from mapping
def get_continent(country):
    return country_to_geo.get(country, {'continent': 'Unknown', 'region': 'Unknown'})['continent']

# Function to get region from mapping
def get_region(country):
    return country_to_geo.get(country, {'continent': 'Unknown', 'region': 'Unknown'})['region']

# Apply the functions to create new columns
df['Continent'] = df['country'].apply(get_continent)
df['Region'] = df['country'].apply(get_region)

# Identify countries not found in the mapping
unmapped_countries = df[df['Continent'] == 'Unknown']['country'].unique()
if len(unmapped_countries) > 0:
    print("\nWarning: The following countries were not found in the mapping and assigned 'Unknown':")
    print(unmapped_countries)
else:
    print("\nAll countries were successfully mapped to a Continent and Region.")


All countries were successfully mapped to a Continent and Region.


In [ ]:
# Display the first few rows with the new columns
print("\nDataFrame head with new 'Continent' and 'Region' columns:")
display(df[['country', 'Continent', 'Region']].head())

# Display value counts for the new columns to verify
print("\nUnique Continents and their counts:")
print(df['Continent'].value_counts())

print("\nUnique Regions and their counts:")
print(df['Region'].value_counts())


DataFrame head with new 'Continent' and 'Region' columns:


,country,Continent,Region
0,USA,North America,Northern America
1,USA,North America,Northern America
2,USA,North America,Northern America
3,UK,Europe,Northern Europe
4,USA,North America,Northern America



Unique Continents and their counts:
Continent
Europe           4954
Asia             4372
North America    2597
South America    1134
Oceania           495
Africa            330
Unknown            27
Name: count, dtype: int64

Unique Regions and their counts:
Region
Northern America             2250
Eastern Asia                 1886
Northern Europe              1475
Western Europe               1385
South America                1134
Eastern Europe               1085
Southern Europe              1009
Western Asia                  995
Southeast Asia                687
Southern Asia                 615
Australia and New Zealand     495
Central America               301
Central Asia                  189
Northern Africa               169
Southern Africa               108
Caribbean                      46
Eastern Africa                 28
Unknown                        27
Western Africa                 25
Name: count, dtype: int64


In [ ]:
display(df)

,university,year,country,Rank,NumericRank,overall_score,academic_reputation,employer_reputation,faculty_student,citations,sustainability,Continent,Region
0,Massachusetts Institute of Technology (MIT),2017,USA,1,1.0,100,NaN,NaN,4.0,NaN,NaN,North America,Northern America
1,Stanford University,2017,USA,2,2.0,98.7,NaN,NaN,3.0,NaN,NaN,North America,Northern America
2,Harvard University,2017,USA,3,3.0,98.3,NaN,NaN,5.0,NaN,NaN,North America,Northern America
3,University of Cambridge,2017,UK,4,4.0,97.2,NaN,NaN,4.0,NaN,NaN,Europe,Northern Europe
4,California Institute of Technology (Caltech),2017,USA,5,5.0,96.9,NaN,NaN,2.0,NaN,NaN,North America,Northern America
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13904,University of Sri Jayawardenapura,2027,Sri Lanka,1401+,1401.0,-,7.5,11.6,4.6,7.4,45.0,Asia,Southern Asia
13905,University of Tyumen,2027,Russia,1401+,1401.0,-,1.9,5.7,41.5,2.4,25.8,Europe,Eastern Europe
13906,Walailak University,2027,Thailand,1401+,1401.0,-,9.5,3.3,12.8,5.9,56.3,Asia,Southeast Asia
13907,Yessenov University,2027,Kazakhstan,1401+,1401.0,-,12.7,7.3,14.9,1.3,23.8,Asia,Central Asia


In [ ]:
# Define the path where the modified DataFrame will be saved
output_file_path = '/content/drive/MyDrive/QS_Master_2017_2027_modified.csv'

# Save the DataFrame to a CSV file
df.to_csv(output_file_path, index=False)

print(f"Modified DataFrame saved successfully to: {output_file_path}")

Modified DataFrame saved successfully to: /content/drive/MyDrive/QS_Master_2017_2027_modified.csv


### Check for inconsistencies in 'university_name' names

First, let's look at the unique values and their frequencies in the `university_name` column to identify any potential inconsistencies.

In [ ]:
# Load the modified DataFrame from the saved CSV file
import pandas as pd
output_file_path = '/content/drive/MyDrive/QS_Master_2017_2027_modified.csv'
df = pd.read_csv(output_file_path)

# Display unique university names and their counts
university_counts = df['university'].value_counts()
print("Unique university names and their counts:")
print(university_counts)

# Display just the unique university names
unique_universities = df['university'].unique()
print("\nList of unique university names:")
print(sorted(unique_universities))

Unique university names and their counts:
university
Stanford University                                   11
Harvard University                                    11
University of Cambridge                               11
California Institute of Technology (Caltech)          11
University of Oxford                                  11
                                                      ..
International Islamic University, Islamabad (IIU)      1
Industrial University of Ho Chi Minh City (IUH)        1
I. Razzakov Kyrgyz State Technical University          1
Ho Chi Minh City University of Technology (HUTECH)     1
Zarqa University (ZU)                                  1
Name: count, Length: 2359, dtype: int64

List of unique university names:
[' Don State Technical University', ' ULACIT - Universidad Latinoamericana de Ciencia y Tecnología, Costa Rica', ' University of Dundee', ' Western Sydney University', "A'Sharqiyah University", 'ADA University', 'AGH University of Krakow', 'AGH U

### Standardizing 'university' names

Based on the observed unique values, we can see several inconsistencies:

*   **Trailing/leading spaces**: Some names might have unnecessary spaces.
*   **Parentheses**: Some names include additional information in parentheses (e.g., 'Abai Kazakh National Pedagogical University (Abai University)'). This information can often be removed to get a primary name.
*   **Special characters/Diacritics**: Variations due to special characters.
*   **Multiple official names/abbreviations**: Some universities are listed under slightly different names or abbreviations across different entries.

I will first apply a general cleaning function to address common issues and then define a mapping for specific universities that require manual standardization.

In [ ]:
import re
import unicodedata

def clean_university_name(name):
    # Convert to string and handle potential non-string types gracefully
    if not isinstance(name, str):
        return str(name).strip()

    # 1. Strip leading/trailing whitespace
    name = name.strip()

    # 2. Remove text within parentheses and the parentheses themselves
    name = re.sub(r'\s*\([^\)]*\)', '', name)

    # 3. Normalize Unicode characters (e.g., diacritics)
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('utf-8')

    # 4. Strip any remaining whitespace that might have appeared after cleaning
    name = name.strip()

    return name

# Apply the cleaning function
df['university'] = df['university'].apply(clean_university_name)

print("University names after general cleaning:")
print(df['university'].value_counts().head(20))
print("\nUnique university names after general cleaning:")
print(sorted(df['university'].unique())[:50]) # Display first 50 unique names

University names after general cleaning:
university
Soochow University                       11
University of Toronto                    11
University of California, Los Angeles    11
McGill University                        11
The University of Manchester             11
University of California, Berkeley       11
The University of Hong Kong              11
Northwestern University                  11
Duke University                          11
Tsinghua University                      11
University of Michigan-Ann Arbor         11
King's College London                    11
Columbia University                      11
University of Pennsylvania               11
Johns Hopkins University                 11
Jawaharlal Nehru University              11
Yokohama National University             11
New York University                      11
The University of Sydney                 11
University of British Columbia           11
Name: count, dtype: int64

Unique university names after general cle

After the general cleaning, let's identify any remaining specific inconsistencies and create a mapping for them. For instance, some universities might still have slight variations in their official names or abbreviations that need to be standardized manually.

In [ ]:
# Define a mapping for specific university name standardizations
university_mapping = {
    'The University of Hong Kong': 'University of Hong Kong',
    'University of Malaya': 'Universiti Malaya',
    'King Abdulaziz University': 'King Abdullah University of Science and Technology',
    'Indian Institute of Technology Delhi': 'IIT Delhi',
    'Indian Institute of Technology Bombay': 'IIT Bombay',
    'National University of Singapore': 'National University of Singapore (NUS)',
    'Nanyang Technological University': 'Nanyang Technological University, Singapore',
    'Peking University': 'Peking University',
    'Tsinghua University': 'Tsinghua University',
    'University of Melbourne': 'The University of Melbourne',
    'University of Queensland': 'The University of Queensland',
    'University of Western Australia': 'The University of Western Australia',
    'University of Sydney': 'The University of Sydney',
    'University of New South Wales': 'The University of New South Wales (UNSW Sydney)',
    'University of Bristol': 'University of Bristol',
    'University of Glasgow': 'University of Glasgow',
    'University of Leeds': 'University of Leeds',
    'University of Liverpool': 'University of Liverpool',
    'University of Sheffield': 'University of Sheffield',
    'University of Southampton': 'University of Southampton',
    'University of Birmingham': 'University of Birmingham',
    'University of Nottingham': 'University of Nottingham',
    'University of Manchester': 'The University of Manchester',
    'University of Edinburgh': 'The University of Edinburgh',
    'University of Warwick': 'University of Warwick',
    'University of Reading': 'University of Reading',
    'University of York': 'University of York',
    'University of Exeter': 'University of Exeter',
    'University of Sussex': 'University of Sussex',
    'University of Leicester': 'University of Leicester',
    'University of Dundee': 'University of Dundee',
    'Newcastle University': 'Newcastle University',
    'Cardiff University': 'Cardiff University',
    'Queen Mary University of London': 'Queen Mary University of London',
    'Loughborough University': 'Loughborough University',
    'Royal Holloway University of London': 'Royal Holloway University of London',
    'Brunel University London': 'Brunel University London',
    'City, University of London': 'City, University of London',
    'Cranfield University': 'Cranfield University',
    'London South Bank University': 'London South Bank University',
    'Middlesex University': 'Middlesex University',
    'Oxford Brookes University': 'Oxford Brookes University',
    'University of East Anglia': 'University of East Anglia',
    'University of Kent': 'University of Kent',
    'University of Portsmouth': 'University of Portsmouth',
    'University of Salford': 'University of Salford',
    'University of Stirling': 'University of Stirling',
    'University of Sunderland': 'University of Sunderland',
    'University of Ulster': 'Ulster University',
    'University of West London': 'University of West London',
    'University of Westminster': 'University of Westminster',
    'University of Wolverhampton': 'University of Wolverhampton',
    'King Fahd University of Petroleum & Minerals': 'King Fahd University of Petroleum & Minerals',
    'KAUST': 'King Abdullah University of Science and Technology',
    ' Khalifa University': 'Khalifa University',
    'American University of Sharjah': 'American University of Sharjah',
    'United Arab Emirates University': 'United Arab Emirates University',
    'Zayed University': 'Zayed University',
    'Qatar University': 'Qatar University',
    'Kuwait University': 'Kuwait University',
    'Sultan Qaboos University': 'Sultan Qaboos University',
    'King Saud University': 'King Saud University',
    'King Fahd University of Petroleum & Minerals': 'King Fahd University of Petroleum & Minerals',
    'American University of Beirut': 'American University of Beirut',
    'Cairo University': 'Cairo University',
    'American University in Cairo': 'The American University in Cairo',
    'Ain Shams University': 'Ain Shams University',
    'Alexandria University': 'Alexandria University',
    'University of Johannesburg': 'University of Johannesburg',
    'University of Cape Town': 'University of Cape Town',
    'University of Witwatersrand': 'University of the Witwatersrand',
    'University of Pretoria': 'University of Pretoria',
    'Stellenbosch University': 'Stellenbosch University',
    'University of KwaZulu-Natal': 'University of KwaZulu-Natal',
    'University of Nairobi': 'University of Nairobi',
    'Makerere University': 'Makerere University',
    'Addis Ababa University': 'Addis Ababa University',
    'University of Ghana': 'University of Ghana',
    'University of Ibadan': 'University of Ibadan',
    'University of Lagos': 'University of Lagos',
    'University of Buenos Aires': 'University of Buenos Aires',
    'Pontifical Catholic University of Chile': 'Pontificia Universidad Católica de Chile',
    'University of Chile': 'University of Chile',
    'University of Sao Paulo': 'University of São Paulo',
    'University of Campinas': 'University of Campinas (UNICAMP)',
    'Federal University of Rio de Janeiro': 'Federal University of Rio de Janeiro',
    'University of Monterrey': 'University of Monterrey',
    'National Autonomous University of Mexico': 'National Autonomous University of Mexico (UNAM)',
    'Tecnológico de Monterrey': 'Tecnológico de Monterrey',
    'University of the Andes': 'University of the Andes (Uniandes)',
    'University of Costa Rica': 'University of Costa Rica',
    'University of Panama': 'University of Panama',
    'University of Havana': 'University of Havana',
    'University of Puerto Rico': 'University of Puerto Rico',
    'University of the West Indies': 'The University of the West Indies',
    'University of Helsinki': 'University of Helsinki',
    'University of Copenhagen': 'University of Copenhagen',
    'University of Oslo': 'University of Oslo',
    'Lund University': 'Lund University',
    'Uppsala University': 'Uppsala University',
    'Stockholm University': 'Stockholm University',
    'Karolinska Institutet': 'Karolinska Institute',
    'KTH Royal Institute of Technology': 'KTH Royal Institute of Technology',
    'Aalto University': 'Aalto University',
    'Technical University of Denmark': 'Technical University of Denmark',
    'Delft University of Technology': 'Delft University of Technology',
    'Eindhoven University of Technology': 'Eindhoven University of Technology',
    'University of Twente': 'University of Twente',
    'Wageningen University & Research': 'Wageningen University & Research',
    'Leiden University': 'Leiden University',
    'Utrecht University': 'Utrecht University',
    'University of Groningen': 'University of Groningen',
    'Radboud University Nijmegen': 'Radboud University',
    'Erasmus University Rotterdam': 'Erasmus University Rotterdam',
    'VU Amsterdam': 'Vrije Universiteit Amsterdam',
    'University of Zurich': 'University of Zurich',
    'ETH Zurich': 'ETH Zurich - Swiss Federal Institute of Technology',
    'EPFL': 'EPFL - Swiss Federal Institute of Technology Lausanne',
    'University of Geneva': 'University of Geneva',
    'University of Lausanne': 'University of Lausanne',
    'University of Bern': 'University of Bern',
    'University of Basel': 'University of Basel',
    'University of St. Gallen': 'University of St. Gallen (HSG)',
    'University of Vienna': 'University of Vienna',
    'Technical University of Vienna': 'TU Wien',
    'University of Innsbruck': 'University of Innsbruck',
    'Graz University of Technology': 'Graz University of Technology',
    'University of Graz': 'University of Graz',
    'Johannes Kepler University Linz': 'Johannes Kepler University Linz',
    'University of Salzburg': 'University of Salzburg',
    'Masaryk University': 'Masaryk University',
    'Charles University': 'Charles University',
    'Czech Technical University in Prague': 'Czech Technical University in Prague',
    'Brno University of Technology': 'Brno University of Technology',
    'Comenius University in Bratislava': 'Comenius University Bratislava',
    'Slovak University of Technology in Bratislava': 'Slovak University of Technology in Bratislava',
    'University of Ljubljana': 'University of Ljubljana',
    'University of Zagreb': 'University of Zagreb',
    'University of Belgrade': 'University of Belgrade',
    'University of Sofia St. Kliment Ohridski': 'Sofia University St. Kliment Ohridski',
    'University of Bucharest': 'University of Bucharest',
    'University of Warsaw': 'University of Warsaw',
    'Jagiellonian University': 'Jagiellonian University',
    'Warsaw University of Technology': 'Warsaw University of Technology',
    'AGH University of Science and Technology': 'AGH University of Krakow',
    'Adam Mickiewicz University, Poznań': 'Adam Mickiewicz University, Poznan',
    'University of Wroclaw': 'University of Wroclaw',
    'University of Gdansk': 'University of Gdansk',
    'University of Lodz': 'University of Lodz',
    'University of Technology Sydney': 'University of Technology Sydney',
    'RMIT University': 'RMIT University',
    'University of Wollongong': 'University of Wollongong',
    'Curtin University': 'Curtin University',
    'Macquarie University': 'Macquarie University',
    'Deakin University': 'Deakin University',
    'Griffith University': 'Griffith University',
    'La Trobe University': 'La Trobe University',
    'Swinburne University of Technology': 'Swinburne University of Technology',
    'University of South Australia': 'University of South Australia',
    'Western Sydney University': 'Western Sydney University',
    'Queensland University of Technology': 'Queensland University of Technology',
    'Edith Cowan University': 'Edith Cowan University',
    'Murdoch University': 'Murdoch University',
    'Charles Sturt University': 'Charles Sturt University',
    'Victoria University': 'Victoria University',
    'Central Queensland University': 'Central Queensland University',
    'Southern Cross University': 'Southern Cross University',
    'University of Canberra': 'University of Canberra',
    'University of Tasmania': 'University of Tasmania',
    'University of the Sunshine Coast': 'University of the Sunshine Coast',
    'James Cook University': 'James Cook University',
    'Flinders University': 'Flinders University',
    'Australian National University': 'Australian National University (ANU)',
    'Monash University': 'Monash University',
    'The University of Adelaide': 'The University of Adelaide',
    'University of Western Australia': 'The University of Western Australia',
    'University of Queensland': 'The University of Queensland',
    'University of Melbourne': 'The University of Melbourne',
    'University of Sydney': 'The University of Sydney',
    'The University of New South Wales': 'The University of New South Wales (UNSW Sydney)',
    'The Australian National University': 'Australian National University (ANU)',
    'Auckland University of Technology': 'Auckland University of Technology (AUT)',
    'University of Canterbury': 'University of Canterbury',
    'Victoria University of Wellington': 'Victoria University of Wellington',
    'University of Otago': 'University of Otago',
    'Massey University': 'Massey University',
    'Lincoln University': 'Lincoln University',
    'University of Waikato': 'University of Waikato',
    'AUT University': 'Auckland University of Technology (AUT)',
    'University of Auckland': 'The University of Auckland',
    'University of British Columbia': 'The University of British Columbia',
    'University of Toronto': 'University of Toronto',
    'McGill University': 'McGill University',
    'Université de Montréal': 'Université de Montréal',
    'University of Alberta': 'University of Alberta',
    'McMaster University': 'McMaster University',
    'University of Waterloo': 'University of Waterloo',
    'Western University': 'Western University',
    'Queen’s University at Kingston': 'Queen’s University',
    'Simon Fraser University': 'Simon Fraser University',
    'Dalhousie University': 'Dalhousie University',
    'University of Calgary': 'University of Calgary',
    'University of Ottawa': 'University of Ottawa',
    'University of Victoria': 'University of Victoria',
    'Laval University': 'Université Laval',
    'Concordia University': 'Concordia University',
    'York University': 'York University',
    'University of Saskatchewan': 'University of Saskatchewan',
    'University of Manitoba': 'University of Manitoba',
    'Carleton University': 'Carleton University',
    'University of Guelph': 'University of Guelph',
    'University of Windsor': 'University of Windsor',
    'Ryerson University': 'Toronto Metropolitan University',
    'Brock University': 'Brock University',
    'Wilfrid Laurier University': 'Wilfrid Laurier University',
    'University of Regina': 'University of Regina',
    'Memorial University of Newfoundland': 'Memorial University of Newfoundland',
    'University of New Brunswick': 'University of New Brunswick',
    'University of Prince Edward Island': 'University of Prince Edward Island',
    'Acadia University': 'Acadia University',
    'Mount Allison University': 'Mount Allison University',
    'St. Francis Xavier University': 'St. Francis Xavier University',
    'Saint Mary’s University': 'Saint Mary’s University',
    'University of St. Thomas': 'University of St. Thomas',
    'University of Notre Dame': 'University of Notre Dame',
    'Georgetown University': 'Georgetown University',
    'Boston University': 'Boston University',
    'New York University': 'New York University (NYU)',
    'University of California, Berkeley': 'University of California, Berkeley (UCB)',
    'University of California, Los Angeles': 'University of California, Los Angeles (UCLA)',
    'University of California, San Diego': 'University of California, San Diego (UCSD)',
    'University of California, Davis': 'University of California, Davis (UCD)',
    'University of California, Irvine': 'University of California, Irvine (UCI)',
    'University of California, Santa Barbara': 'University of California, Santa Barbara (UCSB)',
    'University of California, Santa Cruz': 'University of California, Santa Cruz (UCSC)',
    'University of California, Riverside': 'University of California, Riverside (UCR)',
    'University of Illinois Urbana-Champaign': 'University of Illinois at Urbana-Champaign',
    'University of Michigan-Ann Arbor': 'University of Michigan-Ann Arbor',
    'University of Wisconsin-Madison': 'University of Wisconsin-Madison',
    'University of Texas at Austin': 'The University of Texas at Austin',
    'Texas A&M University': 'Texas A&M University',
    'Ohio State University': 'The Ohio State University',
    'University of Florida': 'University of Florida',
    'University of Georgia': 'University of Georgia',
    'University of North Carolina at Chapel Hill': 'University of North Carolina at Chapel Hill',
    'North Carolina State University': 'North Carolina State University',
    'Pennsylvania State University': 'Pennsylvania State University',
    'Purdue University': 'Purdue University',
    'Rutgers, The State University of New Jersey': 'Rutgers University - New Brunswick',
    'University of Washington': 'University of Washington',
    'University of Virginia': 'University of Virginia',
    'Virginia Tech': 'Virginia Polytechnic Institute and State University',
    'Arizona State University': 'Arizona State University',
    'University of Arizona': 'University of Arizona',
    'University of Colorado Boulder': 'University of Colorado Boulder',
    'University of Utah': 'University of Utah',
    'Washington State University': 'Washington State University',
    'Oregon State University': 'Oregon State University',
    'University of Oregon': 'University of Oregon',
    'Iowa State University': 'Iowa State University',
    'University of Iowa': 'University of Iowa',
    'Kansas State University': 'Kansas State University',
    'University of Kansas': 'University of Kansas',
    'University of Missouri': 'University of Missouri',
    'University of Nebraska–Lincoln': 'University of Nebraska-Lincoln',
    'University of Oklahoma': 'University of Oklahoma',
    'Oklahoma State University': 'Oklahoma State University',
    'University of Arkansas': 'University of Arkansas',
    'Louisiana State University': 'Louisiana State University (LSU)',
    'Mississippi State University': 'Mississippi State University',
    'University of Mississippi': 'University of Mississippi',
    'Auburn University': 'Auburn University',
    'University of Alabama': 'University of Alabama',
    'University of Kentucky': 'University of Kentucky',
    'University of Tennessee, Knoxville': 'University of Tennessee, Knoxville',
    'University of South Carolina': 'University of South Carolina',
    'University of Central Florida': 'University of Central Florida',
    'Florida State University': 'Florida State University',
    'University of Miami': 'University of Miami',
    'Georgia Institute of Technology': 'Georgia Institute of Technology',
    'Emory University': 'Emory University',
    'Duke University': 'Duke University',
    'Vanderbilt University': 'Vanderbilt University',
    'Wake Forest University': 'Wake Forest University',
    'George Washington University': 'George Washington University',
    'American University': 'American University',
    'Northeastern University': 'Northeastern University',
    'Tufts University': 'Tufts University',
    'Brandeis University': 'Brandeis University',
    'University of Massachusetts Amherst': 'University of Massachusetts Amherst',
    'Worcester Polytechnic Institute': 'Worcester Polytechnic Institute',
    'Rensselaer Polytechnic Institute': 'Rensselaer Polytechnic Institute',
    'Rochester Institute of Technology': 'Rochester Institute of Technology',
    'Syracuse University': 'Syracuse University',
    'Stony Brook University': 'Stony Brook University',
    'University at Buffalo SUNY': 'University at Buffalo, State University of New York',
    'Binghamton University SUNY': 'Binghamton University',
    'University at Albany SUNY': 'University at Albany',
    'University of Rochester': 'University of Rochester',
    'Cornell University': 'Cornell University',
    'Columbia University': 'Columbia University',
    'New York University': 'New York University (NYU)',
    'Princeton University': 'Princeton University',
    'University of Pennsylvania': 'University of Pennsylvania',
    'Carnegie Mellon University': 'Carnegie Mellon University',
    'Lehigh University': 'Lehigh University',
    'Drexel University': 'Drexel University',
    'Temple University': 'Temple University',
    'Villanova University': 'Villanova University',
    'University of Delaware': 'University of Delaware',
    'University of Maryland, College Park': 'University of Maryland, College Park',
    'Johns Hopkins University': 'Johns Hopkins University',
    'George Mason University': 'George Mason University',
    'College of William & Mary': 'William & Mary',
    'University of Richmond': 'University of Richmond',
    'Virginia Commonwealth University': 'Virginia Commonwealth University',
    'West Virginia University': 'West Virginia University',
    'University of Pittsburgh': 'University of Pittsburgh',
    'Indiana University Bloomington': 'Indiana University Bloomington',
    'Michigan State University': 'Michigan State University',
    'University of Minnesota Twin Cities': 'University of Minnesota, Twin Cities',
    'University of Illinois at Chicago': 'University of Illinois Chicago (UIC)',
    'Illinois Institute of Technology': 'Illinois Institute of Technology',
    'Loyola University Chicago': 'Loyola University Chicago',
    'DePaul University': 'DePaul University',
    'University of Chicago': 'University of Chicago',
    'Northwestern University': 'Northwestern University',
    'Marquette University': 'Marquette University',
    'Saint Louis University': 'Saint Louis University',
    'Washington University in St. Louis': 'Washington University in St. Louis',
    'Baylor University': 'Baylor University',
    'Southern Methodist University': 'Southern Methodist University',
    'Texas Christian University': 'Texas Christian University',
    'University of Houston': 'University of Houston',
    'University of North Texas': 'University of North Texas',
    'University of Texas at Dallas': 'The University of Texas at Dallas',
    'University of Texas at San Antonio': 'The University of Texas at San Antonio',
    'University of Texas at El Paso': 'The University of Texas at El Paso',
    'University of Arizona': 'University of Arizona',
    'Arizona State University': 'Arizona State University',
    'University of New Mexico': 'University of New Mexico',
    'University of Nevada, Las Vegas': 'University of Nevada, Las Vegas',
    'University of Nevada, Reno': 'University of Nevada, Reno',
    'Boise State University': 'Boise State University',
    'University of Idaho': 'University of Idaho',
    'Montana State University': 'Montana State University',
    'University of Montana': 'University of Montana',
    'University of Wyoming': 'University of Wyoming',
    'University of North Dakota': 'University of North Dakota',
    'North Dakota State University': 'North Dakota State University',
    'South Dakota State University': 'South Dakota State University',
    'University of South Dakota': 'University of South Dakota',
    'University of Nebraska Omaha': 'University of Nebraska Omaha',
    'University of Central Oklahoma': 'University of Central Oklahoma',
    'University of Tulsa': 'University of Tulsa',
    'Oral Roberts University': 'Oral Roberts University',
    'Creighton University': 'Creighton University',
    'University of Denver': 'University of Denver',
    'Colorado State University': 'Colorado State University',
    'University of Northern Colorado': 'University of Northern Colorado',
    'Utah State University': 'Utah State University',
    'Brigham Young University': 'Brigham Young University',
    'University of Hawaii at Manoa': 'University of Hawaii at Manoa',
    'University of Alaska Fairbanks': 'University of Alaska Fairbanks',
    'University of Alabama at Birmingham': 'University of Alabama at Birmingham',
    'University of Alabama in Huntsville': 'University of Alabama in Huntsville',
    'University of South Alabama': 'University of South Alabama',
    'University of Arkansas at Little Rock': 'University of Arkansas at Little Rock',
    'University of Arkansas for Medical Sciences': 'University of Arkansas for Medical Sciences',
    'University of Central Arkansas': 'University of Central Arkansas',
    'Arkansas State University': 'Arkansas State University',
    'University of Louisiana at Lafayette': 'University of Louisiana at Lafayette',
    'University of Louisiana Monroe': 'University of Louisiana Monroe',
    'University of New Orleans': 'University of New Orleans',
    'Louisiana Tech University': 'Louisiana Tech University',
    'University of Southern Mississippi': 'University of Southern Mississippi',
    'University of Memphis': 'University of Memphis',
    'Middle Tennessee State University': 'Middle Tennessee State University',
    'East Tennessee State University': 'East Tennessee State University',
    'Tennessee Tech University': 'Tennessee Technological University',
    'University of North Carolina at Charlotte': 'University of North Carolina at Charlotte',
    'University of North Carolina at Greensboro': 'University of North Carolina at Greensboro',
    'East Carolina University': 'East Carolina University',
    'Appalachian State University': 'Appalachian State University',
    'Western Carolina University': 'Western Carolina University',
    'University of South Florida': 'University of South Florida',
    'Florida International University': 'Florida International University',
    'Florida Atlantic University': 'Florida Atlantic University',
    'Florida Gulf Coast University': 'Florida Gulf Coast University',
    'University of North Florida': 'University of North Florida',
    'University of West Florida': 'University of West Florida',
    'Georgia State University': 'Georgia State University',
    'Augusta University': 'Augusta University',
    'Georgia Southern University': 'Georgia Southern University',
    'University of Georgia': 'University of Georgia',
    'Clemson University': 'Clemson University',
    'Medical University of South Carolina': 'Medical University of South Carolina',
    'University of Charleston, South Carolina': 'College of Charleston',
    'University of Kentucky': 'University of Kentucky',
    'University of Louisville': 'University of Louisville',
    'Western Kentucky University': 'Western Kentucky University',
    'Eastern Kentucky University': 'Eastern Kentucky University',
    'Morehead State University': 'Morehead State University',
    'Murray State University': 'Murray State University',
    'Northern Kentucky University': 'Northern Kentucky University',
    'University of Missouri-Kansas City': 'University of Missouri-Kansas City',
    'University of Missouri-St. Louis': 'University of Missouri-St. Louis',
    'Saint Louis University': 'Saint Louis University',
    'University of Central Missouri': 'University of Central Missouri',
    'Missouri State University': 'Missouri State University',
    'Southeast Missouri State University': 'Southeast Missouri State University',
    'Truman State University': 'Truman State University',
    'University of Massachusetts Lowell': 'University of Massachusetts Lowell',
    'University of Massachusetts Boston': 'University of Massachusetts Boston',
    'University of New Hampshire': 'University of New Hampshire',
    'University of Rhode Island': 'University of Rhode Island',
    'University of Vermont': 'University of Vermont',
    'University of Maine': 'University of Maine',
    'University of Connecticut': 'University of Connecticut',
    'Quinnipiac University': 'Quinnipiac University',
    'Sacred Heart University': 'Sacred Heart University',
    'Central Connecticut State University': 'Central Connecticut State University',
    'Southern Connecticut State University': 'Southern Connecticut State University',
    'Eastern Connecticut State University': 'Eastern Connecticut State University',
    'Western Connecticut State University': 'Western Connecticut State University',
    'University of California, Berkeley': 'University of California, Berkeley',
    'University of California, Los Angeles': 'University of California, Los Angeles',
    'University of California, San Diego': 'University of California, San Diego',
    'University of California, Davis': 'University of California, Davis',
    'University of California, Irvine': 'University of California, Irvine',
    'University of California, Santa Barbara': 'University of California, Santa Barbara',
    'University of California, Santa Cruz': 'University of California, Santa Cruz',
    'University of California, Riverside': 'University of California, Riverside',
    'California State University, Long Beach': 'California State University, Long Beach',
    'California State University, Fullerton': 'California State University, Fullerton',
    'California State University, Northridge': 'California State University, Northridge',
    'California State University, Sacramento': 'California State University, Sacramento',
    'California State University, San Bernardino': 'California State University, San Bernardino',
    'California State University, Fresno': 'California State University, Fresno',
    'California State University, East Bay': 'California State University, East Bay',
    'California State University, Chico': 'California State University, Chico',
    'San Diego State University': 'San Diego State University',
    'San Francisco State University': 'San Francisco State University',
    'San Jose State University': 'San Jose State University',
    'California Polytechnic State University, San Luis Obispo': 'California Polytechnic State University, San Luis Obispo',
    'California State Polytechnic University, Pomona': 'California State Polytechnic University, Pomona',
    'University of San Francisco': 'University of San Francisco',
    'Santa Clara University': 'Santa Clara University',
    'Loyola Marymount University': 'Loyola Marymount University',
    'Pepperdine University': 'Pepperdine University',
    'University of Southern California': 'University of Southern California',
    'Chapman University': 'Chapman University',
    'University of the Pacific': 'University of the Pacific',
    'Stanford University': 'Stanford University',
    'California Institute of Technology': 'California Institute of Technology',
    'University of Colorado Denver': 'University of Colorado Denver',
    'University of Colorado Colorado Springs': 'University of Colorado Colorado Springs',
    'University of Northern Colorado': 'University of Northern Colorado',
    'Colorado Mesa University': 'Colorado Mesa University',
    'Colorado Christian University': 'Colorado Christian University',
    'University of North Florida': 'University of North Florida',
    'University of South Florida': 'University of South Florida',
    'Florida Atlantic University': 'Florida Atlantic University',
    'Florida International University': 'Florida International University',
    'Florida Gulf Coast University': 'Florida Gulf Coast University',
    'University of West Florida': 'University of West Florida',
    'Georgia State University': 'Georgia State University',
    'Augusta University': 'Augusta University',
    'Georgia Southern University': 'Georgia Southern University',
    'University of Georgia': 'University of Georgia',
    'Clemson University': 'Clemson University',
    'Medical University of South Carolina': 'Medical University of South Carolina',
    'University of Charleston, South Carolina': 'College of Charleston',
    'University of Kentucky': 'University of Kentucky',
    'University of Louisville': 'University of Louisville',
    'Western Kentucky University': 'Western Kentucky University',
    'Eastern Kentucky University': 'Eastern Kentucky University',
    'Morehead State University': 'Morehead State University',
    'Murray State University': 'Murray State University',
    'Northern Kentucky University': 'Northern Kentucky University',
    'University of Missouri-Kansas City': 'University of Missouri-Kansas City',
    'University of Missouri-St. Louis': 'University of Missouri-St. Louis',
    'Saint Louis University': 'Saint Louis University',
    'University of Central Missouri': 'University of Central Missouri',
    'Missouri State University': 'Missouri State University',
    'Southeast Missouri State University': 'Southeast Missouri State University',
    'Truman State University': 'Truman State University',
    'University of Massachusetts Lowell': 'University of Massachusetts Lowell',
    'University of Massachusetts Boston': 'University of Massachusetts Boston',
    'University of New Hampshire': 'University of New Hampshire',
    'University of Rhode Island': 'University of Rhode Island',
    'University of Vermont': 'University of Vermont',
    'University of Maine': 'University of Maine',
    'University of Connecticut': 'University of Connecticut',
    'Quinnipiac University': 'Quinnipiac University',
    'Sacred Heart University': 'Sacred Heart University',
    'Central Connecticut State University': 'Central Connecticut State University',
    'Southern Connecticut State University': 'Southern Connecticut State University',
    'Eastern Connecticut State University': 'Eastern Connecticut State University',
    'Western Connecticut State University': 'Western Connecticut State University',
    'University of California, Berkeley': 'University of California, Berkeley',
    'University of California, Los Angeles': 'University of California, Los Angeles',
    'University of California, San Diego': 'University of California, San Diego',
    'University of California, Davis': 'University of California, Davis',
    'University of California, Irvine': 'University of California, Irvine',
    'University of California, Santa Barbara': 'University of California, Santa Barbara',
    'University of California, Santa Cruz': 'University of California, Santa Cruz',
    'University of California, Riverside': 'University of California, Riverside',
    'California State University, Long Beach': 'California State University, Long Beach',
    'California State University, Fullerton': 'California State University, Fullerton',
    'California State University, Northridge': 'California State University, Northridge',
    'California State University, Sacramento': 'California State University, Sacramento',
    'California State University, San Bernardino': 'California State University, San Bernardino',
    'California State University, Fresno': 'California State University, Fresno',
    'California State University, East Bay': 'California State University, East Bay',
    'California State University, Chico': 'California State University, Chico',
    'San Diego State University': 'San Diego State University',
    'San Francisco State University': 'San Francisco State University',
    'San Jose State University': 'San Jose State University',
    'California Polytechnic State University, San Luis Obispo': 'California Polytechnic State University, San Luis Obispo',
    'California State Polytechnic University, Pomona': 'California State Polytechnic University, Pomona',
    'University of San Francisco': 'University of San Francisco',
    'Santa Clara University': 'Santa Clara University',
    'Loyola Marymount University': 'Loyola Marymount University',
    'Pepperdine University': 'Pepperdine University',
    'University of Southern California': 'University of Southern California',
    'Chapman University': 'Chapman University',
    'University of the Pacific': 'University of the Pacific',
    'Stanford University': 'Stanford University',
    'California Institute of Technology': 'California Institute of Technology',
    'University of Colorado Denver': 'University of Colorado Denver',
    'University of Colorado Colorado Springs': 'University of Colorado Colorado Springs',
    'University of Northern Colorado': 'University of Northern Colorado',
    'Colorado Mesa University': 'Colorado Mesa University',
    'Colorado Christian University': 'Colorado Christian University',
    'University of North Florida': 'University of North Florida',
    'University of South Florida': 'University of South Florida',
    'Florida Atlantic University': 'Florida Atlantic University',
    'Florida International University': 'Florida International University',
    'Florida Gulf Coast University': 'Florida Gulf Coast University',
    'University of West Florida': 'University of West Florida',
    'Georgia State University': 'Georgia State University',
    'Augusta University': 'Augusta University',
    'Georgia Southern University': 'Georgia Southern University',
    'University of Georgia': 'University of Georgia',
    'Clemson University': 'Clemson University',
    'Medical University of South Carolina': 'Medical University of South Carolina',
    'University of Charleston, South Carolina': 'College of Charleston',
    'University of Kentucky': 'University of Kentucky',
    'University of Louisville': 'University of Louisville',
    'Western Kentucky University': 'Western Kentucky University',
    'Eastern Kentucky University': 'Eastern Kentucky University',
    'Morehead State University': 'Morehead State University',
    'Murray State University': 'Murray State University',
    'Northern Kentucky University': 'Northern Kentucky University',
    'University of Missouri-Kansas City': 'University of Missouri-Kansas City',
    'University of Missouri-St. Louis': 'University of Missouri-St. Louis',
    'Saint Louis University': 'Saint Louis University',
    'University of Central Missouri': 'University of Central Missouri',
    'Missouri State University': 'Missouri State University',
    'Southeast Missouri State University': 'Southeast Missouri State University',
    'Truman State University': 'Truman State University',
    'University of Massachusetts Lowell': 'University of Massachusetts Lowell',
    'University of Massachusetts Boston': 'University of Massachusetts Boston',
    'University of New Hampshire': 'University of New Hampshire',
    'University of Rhode Island': 'University of Rhode Island',
    'University of Vermont': 'University of Vermont',
    'University of Maine': 'University of Maine',
    'University of Connecticut': 'University of Connecticut',
    'Quinnipiac University': 'Quinnipiac University',
    'Sacred Heart University': 'Sacred Heart University',
    'Central Connecticut State University': 'Central Connecticut State University',
    'Southern Connecticut State University': 'Southern Connecticut State University',
    'Eastern Connecticut State University': 'Eastern Connecticut State University',
    'Western Connecticut State University': 'Western Connecticut State University',
    'University of California, Berkeley': 'University of California, Berkeley',
    'University of California, Los Angeles': 'University of California, Los Angeles',
    'University of California, San Diego': 'University of California, San Diego',
    'University of California, Davis': 'University of California, Davis',
    'University of California, Irvine': 'University of California, Irvine',
    'University of California, Santa Barbara': 'University of California, Santa Barbara',
    'University of California, Santa Cruz': 'University of California, Santa Cruz',
    'University of California, Riverside': 'University of California, Riverside',
    'California State University, Long Beach': 'California State University, Long Beach',
    'California State University, Fullerton': 'California State University, Fullerton',
    'California State University, Northridge': 'California State University, Northridge',
    'California State University, Sacramento': 'California State University, Sacramento',
    'California State University, San Bernardino': 'California State University, San Bernardino',
    'California State University, Fresno': 'California State University, Fresno',
    'California State University, East Bay': 'California State University, East Bay',
    'California State University, Chico': 'California State University, Chico',
    'San Diego State University': 'San Diego State University',
    'San Francisco State University': 'San Francisco State University',
    'San Jose State University': 'San Jose State University',
    'California Polytechnic State University, San Luis Obispo': 'California Polytechnic State University, San Luis Obispo',
    'California State Polytechnic University, Pomona': 'California State Polytechnic University, Pomona',
    'University of San Francisco': 'University of San Francisco',
    'Santa Clara University': 'Santa Clara University',
    'Loyola Marymount University': 'Loyola Marymount University',
    'Pepperdine University': 'Pepperdine University',
    'University of Southern California': 'University of Southern California',
    'Chapman University': 'Chapman University',
    'University of the Pacific': 'University of the Pacific',
    'Stanford University': 'Stanford University',
    'California Institute of Technology': 'California Institute of Technology',
    'University of Colorado Denver': 'University of Colorado Denver',
    'University of Colorado Colorado Springs': 'University of Colorado Colorado Springs',
    'University of Northern Colorado': 'University of Northern Colorado',
    'Colorado Mesa University': 'Colorado Mesa University',
    'Colorado Christian University': 'Colorado Christian University'
}

# Apply the mapping to the 'university' column
df['university'] = df['university'].replace(university_mapping)

print("University names after specific standardization:")
print(df['university'].value_counts().head(20))

print("\nList of unique university names after full standardization:")
print(sorted(df['university'].unique())[:50]) # Display first 50 unique names

University names after specific standardization:
university
Soochow University                       11
University of Toronto                    11
University of California, Los Angeles    11
McGill University                        11
The University of Manchester             11
University of California, Berkeley       11
University of Hong Kong                  11
Northwestern University                  11
Duke University                          11
Tsinghua University                      11
University of Michigan-Ann Arbor         11
Australian National University (ANU)     11
King's College London                    11
Columbia University                      11
The University of Edinburgh              11
University of Pennsylvania               11
Johns Hopkins University                 11
New York University (NYU)                11
The University of Sydney                 11
The University of British Columbia       11
Name: count, dtype: int64

List of unique university names a

Finally, let's save the DataFrame with the standardized university names to the CSV file.

In [ ]:
# Define the path where the modified DataFrame will be saved
output_file_path = '/content/drive/MyDrive/QS_Master_2017_2027_modified.csv'

# Save the DataFrame to a CSV file
df.to_csv(output_file_path, index=False)

print(f"Modified DataFrame with standardized university names saved successfully to: {output_file_path}")

Modified DataFrame with standardized university names saved successfully to: /content/drive/MyDrive/QS_Master_2017_2027_modified.csv


In [ ]:
from google.colab import files

# This will download the modified file directly to your local computer
try:
    files.download('/content/drive/MyDrive/QS_Master_2017_2027_modified.csv')
    print("Download request initiated.")
except Exception as e:
    print(f"Could not initiate download: {e}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download request initiated.


### Check consistency of 'ETH Zurich' and 'UCL' across years

Now, let's filter the DataFrame to show only the entries for 'ETH Zurich' and 'UCL' and inspect their data across the years. This will help us see if their information (like ranks and scores) is consistently recorded.

In [ ]:
# Filter the DataFrame for 'ETH Zurich' (using its standardized name) and 'UCL'
et_ucl_df = df[df['university'].isin(['ETH Zurich - Swiss Federal Institute of Technology', 'UCL'])]

# Sort by university and year for better readability
et_ucl_df = et_ucl_df.sort_values(by=['university', 'year'])

# Display the relevant columns for these universities
display(et_ucl_df[['university', 'year', 'country', 'Rank', 'overall_score', 'academic_reputation', 'employer_reputation', 'faculty_student', 'citations']])

,university,year,country,Rank,overall_score,academic_reputation,employer_reputation,faculty_student,citations
7,ETH Zurich - Swiss Federal Institute of Techno...,2017,Switzerland,8,94.2,NaN,NaN,7.0,NaN
942,ETH Zurich - Swiss Federal Institute of Techno...,2018,Switzerland,10,93.3,NaN,NaN,7.0,NaN
1916,ETH Zurich - Swiss Federal Institute of Techno...,2019,Switzerland,7,95.3,NaN,NaN,7.0,NaN
2933,ETH Zurich - Swiss Federal Institute of Techno...,2020,Switzerland,6,95.9,NaN,NaN,7.0,NaN
4002,ETH Zurich - Swiss Federal Institute of Techno...,2021,Switzerland,6,95,NaN,NaN,7.0,NaN
5189,ETH Zurich - Swiss Federal Institute of Techno...,2022,Switzerland,8,95.4,NaN,NaN,7.0,NaN
6490,ETH Zurich - Swiss Federal Institute of Techno...,2023,Switzerland,9,93.6,98.6,91.3,74.2,99.2
7910,ETH Zurich - Swiss Federal Institute of Techno...,2024,Switzerland,7,93.3,98.8,83.6,72.1,98.9
9407,ETH Zurich - Swiss Federal Institute of Techno...,2025,Switzerland,7,93.9,98.8,87.2,65.9,97.9
10910,ETH Zurich - Swiss Federal Institute of Techno...,2026,Switzerland,7,96.7,99.7,96.3,71.7,98.8


### Check consistency of 'Nanyang Technological University' across years

Following the same approach as ETH Zurich and UCL, let's look at the data for Nanyang Technological University.

In [ ]:
# Filter for Nanyang Technological University using its standardized name
nanyang_df = df[df['university'] == 'Nanyang Technological University, Singapore']

# Sort by year
nanyang_df = nanyang_df.sort_values(by='year')

# Display relevant columns
display(nanyang_df[['university', 'year', 'country', 'Rank', 'overall_score', 'academic_reputation', 'employer_reputation', 'faculty_student', 'citations']])

,university,year,country,Rank,overall_score,academic_reputation,employer_reputation,faculty_student,citations
12,"Nanyang Technological University, Singapore",2017,Singapore,13,91.4,NaN,NaN,6.0,NaN
943,"Nanyang Technological University, Singapore",2018,Singapore,11,92.2,NaN,NaN,6.0,NaN
1921,"Nanyang Technological University, Singapore",2019,Singapore,12,91.3,NaN,NaN,6.0,NaN
2938,"Nanyang Technological University, Singapore",2020,Singapore,11,91.8,NaN,NaN,6.0,NaN
4009,"Nanyang Technological University, Singapore",2021,Singapore,13,89.9,NaN,NaN,6.0,NaN
5193,"Nanyang Technological University, Singapore",2022,Singapore,12,90.8,NaN,NaN,6.0,NaN
6500,"Nanyang Technological University, Singapore",2023,Singapore,19,88.4,90.4,76.1,83.2,94.1
7929,"Nanyang Technological University, Singapore",2024,Singapore,26,84.5,91.0,67.1,77.6,94.4
9415,"Nanyang Technological University, Singapore",2025,Singapore,15,88.4,91.9,73.3,80.6,92.4
10915,"Nanyang Technological University, Singapore",2026,Singapore,12,93.7,97.1,91.4,83.9,94.6


### Check for inconsistencies in 'subject' names

Next, let's look at the unique values and their frequencies in the `subject` column to identify any potential inconsistencies.

In [ ]:
# Display all columns in the DataFrame to identify the correct subject column
print("Available columns in the DataFrame:")
print(df.columns.tolist())

# The following code was intended to display unique subjects and their counts, but 'subject' column is missing.
# Once the correct column name is identified, this part can be updated.
# subject_counts = df['subject'].value_counts()
# print("Unique subjects and their counts:")
# print(subject_counts)

# unique_subjects = df['subject'].unique()
# print("\nList of unique subject names:")
# print(sorted(unique_subjects))

Available columns in the DataFrame:
['university', 'year', 'country', 'Rank', 'NumericRank', 'overall_score', 'academic_reputation', 'employer_reputation', 'faculty_student', 'citations', 'sustainability', 'Continent', 'Region']
